# Tema 5 - Modelado y Simulación

**Fundamentos de Control - 2º GIERM**

---

## Objetivos de aprendizaje

- Obtener el modelo matemático (EDO) de sistemas físicos mecánicos, eléctricos, hidráulicos y térmicos
- Aplicar el procedimiento de **7 pasos** para linealizar sistemas no lineales
- Obtener la **función de transferencia** a partir de la EDO linealizada
- Simular sistemas no lineales con `solve_ivp` y comparar con el modelo linealizado
- Comprender las **analogías electromecánicas** entre dominios físicos
- Identificar cuándo el modelo linealizado es válido y cuándo diverge del real

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
from scipy import signal
from scipy.integrate import solve_ivp

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - modelos linealizados, límites
COLOR_PUNTO = '#238b45'       # verde - puntos de equilibrio, resultados
COLOR_AUX = '#ff7f00'         # naranja - curvas auxiliares
COLOR_CLARO = '#a6cee3'       # azul claro - zonas, fondos

print('Configuración lista.')

---

## 1. Introducción al modelado de sistemas físicos

**Modelar** un sistema físico consiste en obtener las ecuaciones matemáticas que describen su comportamiento dinámico. En control, necesitamos estas ecuaciones para:

- **Predecir** cómo responderá el sistema ante entradas conocidas
- **Diseñar controladores** que modifiquen el comportamiento del sistema
- **Simular** escenarios sin necesidad de construir prototipos físicos

La mayoría de los sistemas reales son **no lineales**, pero trabajamos con modelos **linealizados** porque disponemos de herramientas analíticas potentes (Laplace, funciones de transferencia, diagramas de Bode, etc.).

**Analogía clave:** linealizar es como aproximar una curva por su recta tangente en un punto. Cerca del punto la aproximación es excelente; lejos, puede fallar.

---

## 2. Procedimiento de modelado en 7 pasos

### 2.1 Los 7 pasos

**Paso 1 — Formular la EDO no lineal:** Aplicar leyes físicas (Newton, Kirchhoff, balance de masa/energía) para obtener la ecuación diferencial que gobierna el sistema.

$$\dot{x} = f(x, u)$$

**Paso 2 — Encontrar el punto de equilibrio:** Resolver $f(x_{eq}, u_{eq}) = 0$ para hallar el estado estacionario.

**Paso 3 — Definir variables incrementales:** Expresar las variables como perturbaciones respecto al equilibrio:

$$\delta x = x - x_{eq}, \qquad \delta u = u - u_{eq}$$

**Paso 4 — Expansión en serie de Taylor:** Expandir $f(x, u)$ alrededor del punto de equilibrio:

$$f(x, u) \approx f(x_{eq}, u_{eq}) + \left.\frac{\partial f}{\partial x}\right|_{eq} \delta x + \left.\frac{\partial f}{\partial u}\right|_{eq} \delta u + \text{términos de orden superior}$$

**Paso 5 — Descartar términos de orden superior:** Como $f(x_{eq}, u_{eq}) = 0$ y los términos cuadráticos y superiores son despreciables cerca del equilibrio:

$$\delta\dot{x} = a \cdot \delta x + b \cdot \delta u$$

donde $a = \left.\frac{\partial f}{\partial x}\right|_{eq}$ y $b = \left.\frac{\partial f}{\partial u}\right|_{eq}$.

**Paso 6 — Aplicar la transformada de Laplace:** Con condiciones iniciales nulas:

$$s \cdot \Delta X(s) = a \cdot \Delta X(s) + b \cdot \Delta U(s)$$

**Paso 7 — Obtener la función de transferencia:**

$$\boxed{G(s) = \frac{\Delta X(s)}{\Delta U(s)} = \frac{b}{s - a}}$$

### 2.2 Resumen en recuadro

| Paso | Acción | Resultado |
|------|--------|-----------|
| 1 | Leyes físicas | EDO no lineal: $\dot{x} = f(x, u)$ |
| 2 | $f(x_{eq}, u_{eq}) = 0$ | Punto de equilibrio |
| 3 | $\delta x = x - x_{eq}$ | Variables incrementales |
| 4 | Taylor 1er orden | Aproximación lineal |
| 5 | Eliminar $O(\delta^2)$ | EDO lineal: $\delta\dot{x} = a\,\delta x + b\,\delta u$ |
| 6 | Laplace | Ecuación algebraica en $s$ |
| 7 | Despejar $G(s)$ | Función de transferencia |

> **Truco para el examen:** si el sistema ya es lineal (todos los términos son lineales en $x$ y $u$), se puede ir directamente del Paso 1 al Paso 6 sin necesidad de linealizar.

---

## 3. Ejemplo mecánico: masa-resorte-amortiguador

### 3.1 Descripción del sistema

Una masa $m$ se conecta a una pared mediante un resorte de constante $k$ y un amortiguador viscoso de coeficiente $b$. Se aplica una fuerza externa $F(t)$.

**Paso 1 — EDO (2ª ley de Newton):**

$$m\ddot{x} + b\dot{x} + kx = F(t)$$

Este sistema ya es **lineal**, así que no necesita linealización.

**Paso 6-7 — Laplace y función de transferencia:**

$$m s^2 X(s) + b s X(s) + k X(s) = F(s)$$

$$\boxed{G(s) = \frac{X(s)}{F(s)} = \frac{1}{ms^2 + bs + k}}$$

### 3.2 Forma canónica

Dividiendo por $k$:

$$G(s) = \frac{1/k}{\frac{m}{k}s^2 + \frac{b}{k}s + 1}$$

Identificando: $\omega_n = \sqrt{k/m}$, $\zeta = \frac{b}{2\sqrt{mk}}$, ganancia estática $K = 1/k$.

$$\boxed{G(s) = \frac{K}{\frac{s^2}{\omega_n^2} + \frac{2\zeta}{\omega_n}s + 1}}$$

In [ ]:
# Diagrama del sistema masa-resorte-amortiguador
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Sistema masa-resorte-amortiguador', fontsize=14, fontweight='bold')

# Pared
ax.plot([0, 0], [0, 5], 'k-', lw=3)
for y in np.arange(0, 5, 0.5):
    ax.plot([0, -0.4], [y, y + 0.3], 'k-', lw=1)

# Resorte (zigzag)
x_spring = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
y_spring = [4.0, 4.0, 4.4, 3.6, 4.4, 3.6, 4.4, 4.0, 4.0]
ax.plot(x_spring, y_spring, color=COLOR_PRINCIPAL, lw=2)
ax.text(2, 4.8, r'$k$', fontsize=14, ha='center', color=COLOR_PRINCIPAL, fontweight='bold')

# Amortiguador (rectángulo con pistón)
ax.plot([0, 1.5], [1.5, 1.5], 'k-', lw=2)
rect = plt.Rectangle((1.5, 1.0), 1.5, 1.0, fill=False, edgecolor=COLOR_RECTA, lw=2)
ax.add_patch(rect)
ax.plot([3.0, 4.0], [1.5, 1.5], 'k-', lw=2)
ax.plot([2.25, 2.25], [1.0, 2.0], color=COLOR_RECTA, lw=2, ls='--')
ax.text(2.25, 0.5, r'$b$', fontsize=14, ha='center', color=COLOR_RECTA, fontweight='bold')

# Masa
masa = FancyBboxPatch((4, 0.5), 2.5, 4, boxstyle='round,pad=0.1',
                       facecolor=COLOR_CLARO, edgecolor='black', lw=2)
ax.add_patch(masa)
ax.text(5.25, 2.5, r'$m$', fontsize=20, ha='center', va='center', fontweight='bold')

# Conexiones resorte y amortiguador a masa
ax.plot([4.0, 4.0], [4.0, 3.5], 'k-', lw=2)
ax.plot([4.0, 4.0], [1.5, 1.5], 'k-', lw=2)

# Fuerza externa
ax.annotate('', xy=(8.5, 2.5), xytext=(6.5, 2.5),
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=3))
ax.text(8.8, 2.5, r'$F(t)$', fontsize=16, va='center', color=COLOR_PUNTO, fontweight='bold')

# Eje x
ax.annotate('', xy=(8.5, -0.3), xytext=(5.25, -0.3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(7, -0.8, r'$x(t)$', fontsize=13, ha='center', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Respuesta al escalón del sistema masa-resorte-amortiguador
m, b_damp, k = 1.0, 0.5, 4.0  # kg, Ns/m, N/m

# Función de transferencia: G(s) = 1 / (ms^2 + bs + k)
num = [1.0]
den = [m, b_damp, k]
sys_mech = signal.TransferFunction(num, den)

t_mech, y_mech = signal.step(sys_mech)

# Parámetros característicos
wn = np.sqrt(k / m)
zeta = b_damp / (2 * np.sqrt(m * k))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_mech, y_mech, color=COLOR_PRINCIPAL, lw=2.5, label=r'Respuesta $x(t)$')
ax.axhline(y=1/k, color=COLOR_RECTA, ls='--', lw=1.5, label=f'Valor final = {1/k:.3f} m')
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel(r'Desplazamiento $x$ (m)')
ax.set_title(f'Respuesta al escalón: masa-resorte-amortiguador '
             f'($m={m}$ kg, $b={b_damp}$ Ns/m, $k={k}$ N/m)\n'
             f'$\\omega_n = {wn:.2f}$ rad/s, $\\zeta = {zeta:.3f}$',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

El sistema con $\zeta = 0.125 < 1$ es **subamortiguado**: oscila antes de estabilizarse en el valor final $x_{\infty} = F/k = 1/4 = 0.25$ m.

---

## 4. Ejemplo eléctrico: circuito RLC serie

### 4.1 Descripción del sistema

Un circuito serie con resistencia $R$, inductancia $L$ y capacitancia $C$, alimentado por una tensión $v(t)$. La salida es la corriente $i(t)$.

**Paso 1 — EDO (Ley de Kirchhoff de tensiones):**

$$v(t) = L\frac{di}{dt} + Ri + \frac{1}{C}\int i \, dt$$

Derivando para eliminar la integral:

$$L\frac{d^2i}{dt^2} + R\frac{di}{dt} + \frac{i}{C} = \frac{dv}{dt}$$

Si la salida es la tensión en el condensador $v_C$, con $i = C \frac{dv_C}{dt}$:

$$LC\ddot{v}_C + RC\dot{v}_C + v_C = v(t)$$

**Paso 6-7 — Función de transferencia** (salida: $v_C$):

$$\boxed{G(s) = \frac{V_C(s)}{V(s)} = \frac{1}{LCs^2 + RCs + 1}}$$

### 4.2 Analogía con el sistema mecánico

La estructura es idéntica al masa-resorte-amortiguador: $L \leftrightarrow m$, $R \leftrightarrow b$, $1/C \leftrightarrow k$.

$$\omega_n = \frac{1}{\sqrt{LC}}, \qquad \zeta = \frac{R}{2}\sqrt{\frac{C}{L}}$$

In [ ]:
# Diagrama del circuito RLC serie
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Circuito RLC serie', fontsize=14, fontweight='bold')

# Fuente de tensión (izquierda)
circle_v = plt.Circle((1, 3), 0.8, fill=False, edgecolor=COLOR_PRINCIPAL, lw=2)
ax.add_patch(circle_v)
ax.text(1, 3, r'$v(t)$', fontsize=13, ha='center', va='center', color=COLOR_PRINCIPAL, fontweight='bold')
ax.text(0.7, 3.9, '+', fontsize=14, color=COLOR_PRINCIPAL, fontweight='bold')
ax.text(0.7, 2.0, r'$-$', fontsize=14, color=COLOR_PRINCIPAL, fontweight='bold')

# Cables y componentes (sentido horario)
# Arriba: fuente -> R
ax.plot([1, 1], [3.8, 5.5], 'k-', lw=2)
ax.plot([1, 3], [5.5, 5.5], 'k-', lw=2)

# R (arriba)
rect_r = plt.Rectangle((3, 5.1), 2, 0.8, fill=False, edgecolor=COLOR_RECTA, lw=2)
ax.add_patch(rect_r)
ax.text(4, 5.5, r'$R$', fontsize=14, ha='center', va='center', color=COLOR_RECTA, fontweight='bold')
ax.plot([5, 7], [5.5, 5.5], 'k-', lw=2)

# L (derecha, bajando)
ax.plot([7, 7], [5.5, 4.5], 'k-', lw=2)
# Inductancia (arcos)
for yy in [4.0, 3.5, 3.0, 2.5]:
    arc = mpatches.Arc((7, yy), 0.8, 0.5, angle=0, theta1=-90, theta2=90, color=COLOR_PUNTO, lw=2)
    ax.add_patch(arc)
ax.text(7.8, 3.25, r'$L$', fontsize=14, color=COLOR_PUNTO, fontweight='bold')
ax.plot([7, 7], [2.25, 1.5], 'k-', lw=2)

# C (abajo)
ax.plot([7, 5.5], [1.5, 1.5], 'k-', lw=2)
ax.plot([5.5, 5.5], [1.0, 2.0], color=COLOR_AUX, lw=3)
ax.plot([4.5, 4.5], [1.0, 2.0], color=COLOR_AUX, lw=3)
ax.text(5, 0.4, r'$C$', fontsize=14, ha='center', color=COLOR_AUX, fontweight='bold')
ax.plot([4.5, 1], [1.5, 1.5], 'k-', lw=2)
ax.plot([1, 1], [1.5, 2.2], 'k-', lw=2)

# Etiqueta v_C
ax.annotate('', xy=(5.5, 0.8), xytext=(4.5, 0.8),
            arrowprops=dict(arrowstyle='<->', color=COLOR_AUX, lw=1.5))
ax.text(5, -0.1, r'$v_C$', fontsize=13, ha='center', color=COLOR_AUX, fontweight='bold')

# Flecha corriente
ax.annotate('', xy=(2.5, 5.5), xytext=(1.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(2, 6.0, r'$i(t)$', fontsize=12, ha='center', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Respuesta al escalón del circuito RLC serie
R, L, C = 100, 0.1, 100e-6  # ohm, henry, farad

# G(s) = 1 / (LCs^2 + RCs + 1)
num_rlc = [1.0]
den_rlc = [L * C, R * C, 1.0]
sys_rlc = signal.TransferFunction(num_rlc, den_rlc)

t_rlc, y_rlc = signal.step(sys_rlc)

wn_rlc = 1 / np.sqrt(L * C)
zeta_rlc = (R / 2) * np.sqrt(C / L)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_rlc, y_rlc, color=COLOR_PRINCIPAL, lw=2.5, label=r'$v_C(t)$ / $V$')
ax.axhline(y=1.0, color=COLOR_RECTA, ls='--', lw=1.5, label='Valor final = 1.0 V')
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel(r'Tensión en el condensador $v_C$ (V)')
ax.set_title(f'Respuesta al escalón: RLC serie '
             f'($R={R}\\;\\Omega$, $L={L}$ H, $C={C*1e6:.0f}\\;\\mu$F)\n'
             f'$\\omega_n = {wn_rlc:.1f}$ rad/s, $\\zeta = {zeta_rlc:.3f}$',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. Ejemplo hidráulico: tanque con válvula no lineal

### 5.1 Modelo no lineal

Un tanque cilíndrico de sección $A$ recibe un caudal de entrada $Q_i$ y descarga por una válvula de salida con ley $Q_o = c\sqrt{h}$, donde $h$ es la altura del líquido.

**Paso 1 — Balance de masa:**

$$A\frac{dh}{dt} = Q_i - c\sqrt{h}$$

**Esta EDO es no lineal** por el término $\sqrt{h}$.

### 5.2 Linealización (7 pasos completos)

**Paso 2 — Equilibrio:** En régimen permanente, $dh/dt = 0$:

$$Q_{i,eq} = c\sqrt{h_{eq}} \implies h_{eq} = \left(\frac{Q_{i,eq}}{c}\right)^2$$

**Paso 3 — Variables incrementales:**

$$\delta h = h - h_{eq}, \qquad \delta Q_i = Q_i - Q_{i,eq}$$

**Paso 4 — Taylor de $\sqrt{h}$ alrededor de $h_{eq}$:**

$$\sqrt{h} \approx \sqrt{h_{eq}} + \frac{1}{2\sqrt{h_{eq}}}\delta h$$

**Paso 5 — EDO linealizada:** Sustituyendo y cancelando términos de equilibrio:

$$A\frac{d(\delta h)}{dt} = \delta Q_i - \frac{c}{2\sqrt{h_{eq}}}\delta h$$

Definiendo la **resistencia hidráulica** $R_h = \frac{2\sqrt{h_{eq}}}{c} = \frac{2h_{eq}}{Q_{i,eq}}$:

$$A\frac{d(\delta h)}{dt} = \delta Q_i - \frac{\delta h}{R_h}$$

**Paso 6-7 — Función de transferencia:**

$$\boxed{G(s) = \frac{\Delta H(s)}{\Delta Q_i(s)} = \frac{R_h}{A R_h s + 1} = \frac{R_h}{\tau s + 1}}$$

donde $\tau = A R_h$ es la constante de tiempo del sistema linealizado.

> **Nota importante:** la resistencia $R_h$ depende del punto de equilibrio $h_{eq}$. Si cambiamos el punto de operación, cambia el modelo linealizado.

In [ ]:
# Diagrama del sistema de tanque hidráulico
fig, ax = plt.subplots(figsize=(8, 7))
ax.set_xlim(-1, 9)
ax.set_ylim(-1, 9)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Sistema de tanque hidráulico con válvula no lineal',
             fontsize=14, fontweight='bold')

# Tanque
ax.plot([2, 2], [1, 7], 'k-', lw=3)  # pared izquierda
ax.plot([6, 6], [1, 7], 'k-', lw=3)  # pared derecha
ax.plot([2, 6], [1, 1], 'k-', lw=3)  # fondo

# Nivel de agua
agua = plt.Rectangle((2, 1), 4, 3, facecolor=COLOR_CLARO, alpha=0.5)
ax.add_patch(agua)

# Flecha h
ax.annotate('', xy=(1.3, 4), xytext=(1.3, 1),
            arrowprops=dict(arrowstyle='<->', color=COLOR_PRINCIPAL, lw=2))
ax.text(0.7, 2.5, r'$h(t)$', fontsize=14, color=COLOR_PRINCIPAL, fontweight='bold')

# Caudal entrada (arriba)
ax.annotate('', xy=(4, 7), xytext=(4, 8.5),
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=3))
ax.text(4.3, 8, r'$Q_i$', fontsize=15, color=COLOR_PUNTO, fontweight='bold')

# Válvula y salida (abajo derecha)
ax.plot([6, 7], [1.5, 1.5], 'k-', lw=2)
# Válvula (triángulo doble)
valve_x = [7, 7.5, 7, 7]
valve_y = [1.1, 1.5, 1.9, 1.1]
ax.fill(valve_x, valve_y, facecolor='white', edgecolor=COLOR_RECTA, lw=2)
valve_x2 = [8, 7.5, 8, 8]
valve_y2 = [1.1, 1.5, 1.9, 1.1]
ax.fill(valve_x2, valve_y2, facecolor='white', edgecolor=COLOR_RECTA, lw=2)

# Caudal salida
ax.annotate('', xy=(8.5, 1.5), xytext=(8, 1.5),
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=3))
ax.text(7, 0.3, r'$Q_o = c\sqrt{h}$', fontsize=13, ha='center',
        color=COLOR_RECTA, fontweight='bold')

# Sección A
ax.annotate('', xy=(6, 6.5), xytext=(2, 6.5),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.text(4, 6.8, 'Sección $A$', fontsize=12, ha='center', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Comparación: modelo no lineal (solve_ivp) vs linealizado
A_tank = 2.0    # m^2
c_valve = 0.5   # m^(5/2)/s
Qi_eq = 1.0     # m^3/s - caudal de equilibrio
h_eq = (Qi_eq / c_valve)**2  # = 4.0 m

# Perturbación pequeña en el caudal de entrada
dQi = 0.1  # m^3/s

# Parámetros del modelo linealizado
Rh = 2 * np.sqrt(h_eq) / c_valve  # resistencia hidráulica
tau = A_tank * Rh                   # constante de tiempo

# --- Simulación no lineal con solve_ivp ---
def tanque_nl(t, h, Qi_total):
    """EDO no lineal del tanque."""
    dhdt = (Qi_total - c_valve * np.sqrt(max(h[0], 0))) / A_tank
    return [dhdt]

t_span = (0, 60)
t_eval = np.linspace(0, 60, 500)

sol_nl = solve_ivp(tanque_nl, t_span, [h_eq], t_eval=t_eval,
                   args=(Qi_eq + dQi,), method='RK45')

# --- Modelo linealizado: G(s) = Rh / (tau*s + 1) ---
sys_tank = signal.TransferFunction([Rh], [tau, 1])
t_lin, y_lin = signal.step(sys_tank, T=t_eval)
# Escalar por dQi y sumar equilibrio
h_lin = h_eq + dQi * y_lin

# --- Gráfica comparativa ---
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sol_nl.t, sol_nl.y[0], color=COLOR_PRINCIPAL, lw=2.5,
        label='No lineal (solve_ivp)')
ax.plot(t_lin, h_lin, color=COLOR_RECTA, lw=2, ls='--',
        label='Linealizado')
ax.axhline(y=h_eq, color='gray', ls=':', lw=1, label=f'Equilibrio $h_{{eq}} = {h_eq:.1f}$ m')
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel(r'Altura $h$ (m)')
ax.set_title(f'Tanque hidráulico: no lineal vs linealizado '
             f'($\\Delta Q_i = {dQi}$ m\u00b3/s, perturbación pequeña)',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Parámetros: h_eq = {h_eq:.2f} m, Rh = {Rh:.2f} s/m\u00b2, tau = {tau:.2f} s')

Con una perturbación pequeña ($\Delta Q_i = 0.1$ m$^3$/s, un 10% del caudal de equilibrio), ambos modelos coinciden casi perfectamente. El modelo linealizado es un sistema de **primer orden** con constante de tiempo $\tau = A \cdot R_h$.

---

## 6. Ejemplo térmico: habitación con calefactor

### 6.1 Modelo

Una habitación de capacidad térmica $C_{th}$ se calienta con un calefactor de potencia $P$ y pierde calor al exterior (a temperatura $T_{ext}$) a través de paredes con conductancia térmica $K$.

**Paso 1 — Balance de energía:**

$$C_{th}\frac{dT}{dt} = P - K(T - T_{ext})$$

Este sistema ya es **lineal** en $T$ y $P$ (la temperatura exterior $T_{ext}$ es una perturbación constante).

**Reescribiendo:**

$$C_{th}\frac{dT}{dt} + KT = P + K \cdot T_{ext}$$

**Paso 6-7 — Función de transferencia** (entrada: $P$, salida: $T$, con $T_{ext}$ constante):

Usando variables incrementales $\delta T = T - T_{eq}$, $\delta P = P - P_{eq}$:

$$\boxed{G(s) = \frac{\Delta T(s)}{\Delta P(s)} = \frac{1/K}{\frac{C_{th}}{K}s + 1} = \frac{R_{th}}{\tau_{th} s + 1}}$$

donde $R_{th} = 1/K$ es la resistencia térmica y $\tau_{th} = C_{th}/K$ es la constante de tiempo térmica.

**Equilibrio:** $T_{eq} = T_{ext} + P_{eq}/K$

### 6.2 Analogía eléctrica

Es análogo a un circuito RC: $C_{th} \leftrightarrow C$, $K \leftrightarrow 1/R$, $T \leftrightarrow v_C$, $P \leftrightarrow i$.

In [ ]:
# Respuesta al escalón del sistema térmico
C_th = 5e5    # J/K - capacidad térmica de la habitación
K_th = 50     # W/K - conductancia térmica de las paredes
T_ext = 5     # °C - temperatura exterior
P_eq = 1000   # W - potencia del calefactor en equilibrio

T_eq = T_ext + P_eq / K_th  # = 25 °C
R_th = 1 / K_th
tau_th = C_th / K_th  # constante de tiempo

# Incremento de potencia
dP = 500  # W

# Modelo linealizado: G(s) = R_th / (tau_th * s + 1)
sys_thermal = signal.TransferFunction([R_th], [tau_th, 1])
t_th = np.linspace(0, 80000, 1000)
t_th_resp, y_th_resp = signal.step(sys_thermal, T=t_th)

T_resp = T_eq + dP * y_th_resp

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_th_resp / 3600, T_resp, color=COLOR_PRINCIPAL, lw=2.5,
        label=r'$T(t)$')
ax.axhline(y=T_eq, color='gray', ls=':', lw=1, label=f'$T_{{eq}}$ = {T_eq:.0f} °C')
ax.axhline(y=T_eq + dP * R_th, color=COLOR_RECTA, ls='--', lw=1.5,
           label=f'Nuevo equilibrio = {T_eq + dP * R_th:.0f} °C')
ax.axvline(x=tau_th / 3600, color=COLOR_PUNTO, ls='--', lw=1,
           label=f'$\\tau$ = {tau_th/3600:.1f} h')
ax.set_xlabel('Tiempo (horas)')
ax.set_ylabel(r'Temperatura $T$ (°C)')
ax.set_title(f'Respuesta térmica: incremento de potencia $\\Delta P = {dP}$ W\n'
             f'$C_{{th}} = {C_th:.0e}$ J/K, $K = {K_th}$ W/K, '
             f'$\\tau = {tau_th/3600:.1f}$ h',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'T_eq = {T_eq:.0f} °C, tau = {tau_th:.0f} s = {tau_th/3600:.1f} h')

---

## 7. Comparación no lineal vs lineal: efecto del tamaño de la perturbación

El modelo linealizado es una aproximación local. Veamos cómo se comporta con perturbaciones de distinto tamaño usando el ejemplo del tanque hidráulico.

In [ ]:
# Comparación NL vs lineal: perturbación pequeña vs grande
perturbaciones = [0.05, 0.2, 0.8]  # m^3/s
etiquetas = ['Pequeña', 'Media', 'Grande']
colores_nl = [COLOR_PRINCIPAL, COLOR_PUNTO, COLOR_AUX]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (dQ, etiq) in enumerate(zip(perturbaciones, etiquetas)):
    ax = axes[idx]

    # No lineal
    sol = solve_ivp(tanque_nl, (0, 80), [h_eq], t_eval=np.linspace(0, 80, 500),
                    args=(Qi_eq + dQ,), method='RK45')

    # Linealizado
    t_l, y_l = signal.step(sys_tank, T=np.linspace(0, 80, 500))
    h_l = h_eq + dQ * y_l

    ax.plot(sol.t, sol.y[0], color=colores_nl[idx], lw=2.5, label='No lineal')
    ax.plot(t_l, h_l, color=COLOR_RECTA, lw=2, ls='--', label='Linealizado')
    ax.axhline(y=h_eq, color='gray', ls=':', lw=1)

    # Error final
    h_final_nl = sol.y[0][-1]
    h_final_lin = h_l[-1]
    error_pct = abs(h_final_nl - h_final_lin) / h_final_nl * 100

    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel(r'$h$ (m)')
    ax.set_title(f'{etiq}: $\\Delta Q_i = {dQ}$ m\u00b3/s ({dQ/Qi_eq*100:.0f}%)\n'
                 f'Error final: {error_pct:.1f}%', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Validez del modelo linealizado según el tamaño de la perturbación',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Conclusiones clave:**

- Con perturbaciones **pequeñas** (5-10% del valor de equilibrio), el modelo linealizado es una excelente aproximación
- Con perturbaciones **grandes** (>50%), el error se hace significativo
- El modelo no lineal siempre llega a un equilibrio **diferente** al que predice el linealizado para perturbaciones grandes
- **Regla práctica:** el modelo linealizado es fiable para perturbaciones de hasta un 10-20% del valor de equilibrio

---

## 8. Analogías electromecánicas

Las ecuaciones que gobiernan sistemas de distintos dominios físicos tienen la **misma estructura matemática**. Esto permite trasladar conocimiento de un dominio a otro.

### 8.1 Analogía fuerza-tensión

| Mecánico (traslación) | Eléctrico | Hidráulico | Térmico |
|------------------------|----------|------------|----------|
| Fuerza $F$ | Tensión $v$ | Presión $P$ (o $\rho g h$) | Temperatura $T$ |
| Velocidad $\dot{x}$ | Corriente $i$ | Caudal $Q$ | Flujo de calor $\dot{Q}$ |
| Desplazamiento $x$ | Carga $q$ | Volumen $V$ | Calor $Q$ |
| Masa $m$ | Inductancia $L$ | Inertancia $I = \rho l / A$ | --- |
| Amortiguador $b$ | Resistencia $R$ | Resistencia hidr. $R_h$ | Resistencia térm. $R_{th} = 1/K$ |
| Resorte $1/k$ | Capacitancia $C$ | Capacitancia hidr. $C_h = A$ | Capacidad térm. $C_{th}$ |

### 8.2 Ecuación unificada de 2º orden

Todos los sistemas de segundo orden se describen con:

$$\boxed{\frac{d^2 y}{dt^2} + 2\zeta\omega_n \frac{dy}{dt} + \omega_n^2 y = \omega_n^2 \cdot u(t)}$$

| Parámetro | Mecánico | Eléctrico (RLC) |
|-----------|---------|------------------|
| $\omega_n$ | $\sqrt{k/m}$ | $1/\sqrt{LC}$ |
| $\zeta$ | $b/(2\sqrt{mk})$ | $(R/2)\sqrt{C/L}$ |
| Ganancia DC | $1/k$ | $1$ |

### 8.3 Ecuación unificada de 1er orden

Los sistemas de primer orden siguen:

$$\boxed{\tau \frac{dy}{dt} + y = K \cdot u(t)}$$

| Parámetro | Hidráulico (tanque) | Térmico (habitación) | Eléctrico (RC) |
|-----------|---------------------|---------------------|----------------|
| $\tau$ | $A \cdot R_h$ | $C_{th}/K$ | $RC$ |
| $K$ (ganancia) | $R_h$ | $1/K_{th}$ | $1$ |

---

## 9. Ejercicios resueltos

### Ejercicio 9.1: Péndulo simple — modelado y linealización

**Datos:** Péndulo de longitud $l = 0.5$ m, masa $m = 0.2$ kg, fricción viscosa $b = 0.05$ Nm·s/rad, $g = 9.81$ m/s$^2$.

**Paso 1 — EDO no lineal (2ª ley de Newton rotacional):**

$$ml^2 \ddot{\theta} + b\dot{\theta} + mgl\sin(\theta) = \tau_{ext}$$

**Paso 2 — Equilibrio:** $\theta_{eq} = 0$, $\tau_{ext,eq} = 0$.

**Paso 3 — Variables incrementales:** $\delta\theta = \theta - 0 = \theta$.

**Paso 4 — Taylor de $\sin(\theta)$ alrededor de $\theta = 0$:**

$$\sin(\theta) \approx \theta \quad (\text{para } \theta \text{ pequeño})$$

**Paso 5 — EDO linealizada:**

$$ml^2 \ddot{\theta} + b\dot{\theta} + mgl\theta = \tau_{ext}$$

**Paso 6-7 — Función de transferencia:**

$$G(s) = \frac{\Theta(s)}{T_{ext}(s)} = \frac{1}{ml^2 s^2 + bs + mgl}$$

**Sustitución numérica:**

$$ml^2 = 0.2 \times 0.5^2 = 0.05 \;\text{kg}\cdot\text{m}^2$$

$$mgl = 0.2 \times 9.81 \times 0.5 = 0.981 \;\text{N}\cdot\text{m}$$

$$\boxed{G(s) = \frac{1}{0.05s^2 + 0.05s + 0.981}}$$

**Parámetros característicos:**

$$\omega_n = \sqrt{\frac{mgl}{ml^2}} = \sqrt{\frac{g}{l}} = \sqrt{\frac{9.81}{0.5}} = 4.43 \;\text{rad/s}$$

$$\zeta = \frac{b}{2ml^2 \omega_n} = \frac{0.05}{2 \times 0.05 \times 4.43} = 0.113$$

El sistema es **subamortiguado** ($\zeta < 1$): el péndulo oscila antes de estabilizarse.

### Ejercicio 9.2: Motor DC — modelado electromecánico

**Datos:** $R_a = 2\;\Omega$, $L_a = 0.01$ H, $K_t = 0.1$ Nm/A (constante de par), $K_e = 0.1$ V·s/rad (constante de fuerza contra-electromotriz), $J = 0.01$ kg·m$^2$, $b = 0.001$ Nm·s/rad.

**Paso 1 — Ecuaciones del motor:**

Circuito de armadura:
$$V_a = L_a \frac{di_a}{dt} + R_a i_a + K_e \dot{\theta}$$

Ecuación mecánica:
$$J\ddot{\theta} + b\dot{\theta} = K_t i_a$$

**Paso 6 — Laplace (ambas ecuaciones):**

$$V_a(s) = (L_a s + R_a) I_a(s) + K_e s \Theta(s)$$

$$(Js + b) s \Theta(s) = K_t I_a(s)$$

**Paso 7 — Eliminar $I_a(s)$:**

De la ecuación mecánica: $I_a(s) = \frac{(Js + b) s \Theta(s)}{K_t}$

Sustituyendo en la eléctrica:

$$V_a(s) = (L_a s + R_a) \frac{(Js + b) s \Theta(s)}{K_t} + K_e s \Theta(s)$$

$$\boxed{G(s) = \frac{\Theta(s)}{V_a(s)} = \frac{K_t}{s[(L_a s + R_a)(Js + b) + K_t K_e]}}$$

**Sustitución numérica (despreciando $L_a$ por ser pequeña):**

$$G(s) \approx \frac{K_t}{s[R_a(Js + b) + K_t K_e]} = \frac{0.1}{s[0.02s + 0.002 + 0.01]}$$

$$\boxed{G(s) = \frac{0.1}{s(0.02s + 0.012)} = \frac{8.33}{s(1.667s + 1)}}$$

El motor es un sistema de **tipo 1** (un integrador): la velocidad angular alcanza un valor constante, pero el ángulo crece indefinidamente ante entrada escalón.

### Ejercicio 9.3: Sistema de dos tanques en cascada

**Datos:** Dos tanques idénticos, sección $A = 1$ m$^2$, coeficiente de descarga $c = 0.3$ m$^{5/2}$/s. La salida del tanque 1 alimenta el tanque 2.

**Paso 1 — EDOs no lineales:**

$$A\frac{dh_1}{dt} = Q_i - c\sqrt{h_1}$$

$$A\frac{dh_2}{dt} = c\sqrt{h_1} - c\sqrt{h_2}$$

**Paso 2 — Equilibrio:** Con $Q_{i,eq} = 0.6$ m$^3$/s:

$$c\sqrt{h_{1,eq}} = Q_{i,eq} \implies h_{1,eq} = \left(\frac{0.6}{0.3}\right)^2 = 4 \;\text{m}$$

$$c\sqrt{h_{2,eq}} = c\sqrt{h_{1,eq}} \implies h_{2,eq} = h_{1,eq} = 4 \;\text{m}$$

**Pasos 3-5 — Linealización:**

$$R_{h1} = R_{h2} = \frac{2\sqrt{4}}{0.3} = \frac{4}{0.3} = 13.33 \;\text{s/m}^2$$

$$\tau_1 = \tau_2 = A \cdot R_h = 1 \times 13.33 = 13.33 \;\text{s}$$

**Pasos 6-7 — Funciones de transferencia:**

Tanque 1: $G_1(s) = \dfrac{R_h}{\tau s + 1}$

Tanque 2: $G_2(s) = \dfrac{1}{\tau s + 1}$ (ganancia 1, porque la salida del tanque 1 ya está en m$^3$/s)

$$\boxed{G(s) = \frac{\Delta H_2(s)}{\Delta Q_i(s)} = G_1(s) \cdot G_2(s) = \frac{R_h}{(\tau s + 1)^2}}$$

El sistema total es de **segundo orden sobreamortiguado** (dos polos reales iguales).

### Ejercicio 9.4: Sistema térmico con dos zonas

**Datos:** Dos habitaciones comunicadas. Habitación 1: capacidad $C_1 = 3 \times 10^5$ J/K, calefactor $P$. Habitación 2: capacidad $C_2 = 2 \times 10^5$ J/K, sin calefactor. Conductancia entre ellas: $K_{12} = 30$ W/K. Conductancia al exterior: $K_1 = 40$ W/K, $K_2 = 25$ W/K. $T_{ext} = 0$ °C.

**Paso 1 — Balances de energía:**

$$C_1 \frac{dT_1}{dt} = P - K_1(T_1 - T_{ext}) - K_{12}(T_1 - T_2)$$

$$C_2 \frac{dT_2}{dt} = K_{12}(T_1 - T_2) - K_2(T_2 - T_{ext})$$

**Paso 2 — Equilibrio:** Con $P_{eq} = 2000$ W y $T_{ext} = 0$ °C:

De la ecuación 2: $K_{12}(T_{1,eq} - T_{2,eq}) = K_2 \cdot T_{2,eq} \implies T_{1,eq} = T_{2,eq}(1 + K_2/K_{12})$

$$T_{1,eq} = T_{2,eq} \left(1 + \frac{25}{30}\right) = 1.833 \cdot T_{2,eq}$$

De la ecuación 1: $P_{eq} = K_1 T_{1,eq} + K_{12}(T_{1,eq} - T_{2,eq}) = K_1 T_{1,eq} + K_2 T_{2,eq}$

$$2000 = 40 \times 1.833 T_{2,eq} + 25 T_{2,eq} = (73.33 + 25) T_{2,eq} = 98.33 T_{2,eq}$$

$$\boxed{T_{2,eq} = 20.34 \;\text{°C}, \qquad T_{1,eq} = 37.29 \;\text{°C}}$$

**Paso 6-7 — Forma matricial en Laplace:**

$$\begin{pmatrix} C_1 s + K_1 + K_{12} & -K_{12} \\ -K_{12} & C_2 s + K_2 + K_{12} \end{pmatrix} \begin{pmatrix} \Delta T_1 \\ \Delta T_2 \end{pmatrix} = \begin{pmatrix} \Delta P \\ 0 \end{pmatrix}$$

La función de transferencia $G(s) = \Delta T_2(s) / \Delta P(s)$ se obtiene por Cramer:

$$G(s) = \frac{K_{12}}{(C_1 s + K_1 + K_{12})(C_2 s + K_2 + K_{12}) - K_{12}^2}$$

### Ejercicio 9.5: Identificación de FT a partir de respuesta al escalón experimental

**Datos experimentales:** Se aplica un escalón de amplitud $A = 2$ V a un sistema desconocido. Se observa:
- Valor final de la salida: $y_{\infty} = 0.8$ V
- Tiempo en alcanzar el 63.2% del valor final: $t_{63\%} = 3$ s
- No hay sobreoscilación $\to$ **primer orden**

**Paso 1 — Identificar la ganancia estática:**

$$K = \frac{y_{\infty}}{A} = \frac{0.8}{2} = 0.4$$

**Paso 2 — Identificar la constante de tiempo:**

Para un sistema de primer orden, al 63.2% del valor final se cumple $t = \tau$:

$$\tau = t_{63\%} = 3 \;\text{s}$$

**Paso 3 — Función de transferencia:**

$$\boxed{G(s) = \frac{K}{\tau s + 1} = \frac{0.4}{3s + 1}}$$

**Verificación:**
- $y(\infty) = K \cdot A = 0.4 \times 2 = 0.8$ V $\checkmark$
- $y(\tau) = K \cdot A \cdot (1 - e^{-1}) = 0.8 \times 0.632 = 0.506$ V
- $y(3\tau) = 0.8 \times 0.950 = 0.76$ V (95% del final)
- $y(5\tau) = 0.8 \times 0.993 = 0.794$ V (>99%, régimen permanente)

> **Método alternativo para 2º orden con sobreoscilación:** si se observa sobreoscilación $M_p$ y periodo de oscilación $T_d$, se puede obtener $\zeta$ de $M_p = e^{-\pi\zeta/\sqrt{1-\zeta^2}}$ y $\omega_d = 2\pi/T_d$.

In [ ]:
# Verificación gráfica del ejercicio 9.5
K_id, tau_id, A_step = 0.4, 3.0, 2.0
sys_id = signal.TransferFunction([K_id], [tau_id, 1])
t_id = np.linspace(0, 20, 500)
t_id_resp, y_id_resp = signal.step(sys_id, T=t_id)
y_id_resp = A_step * y_id_resp  # escalar por amplitud del escalón

# Datos "experimentales" simulados con ruido
np.random.seed(42)
t_exp = np.linspace(0, 20, 30)
y_exp = A_step * K_id * (1 - np.exp(-t_exp / tau_id)) + np.random.normal(0, 0.015, len(t_exp))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(t_exp, y_exp, 'o', color=COLOR_PRINCIPAL, ms=6, label='Datos experimentales')
ax.plot(t_id_resp, y_id_resp, color=COLOR_RECTA, lw=2.5,
        label=r'Modelo: $G(s) = \frac{0.4}{3s+1}$')
ax.axhline(y=A_step * K_id, color='gray', ls=':', lw=1,
           label=f'$y_{{\\infty}}$ = {A_step * K_id:.1f} V')
ax.axhline(y=A_step * K_id * 0.632, color=COLOR_PUNTO, ls='--', lw=1,
           label='63.2% del valor final')
ax.axvline(x=tau_id, color=COLOR_PUNTO, ls='--', lw=1,
           label=f'$\\tau$ = {tau_id:.0f} s')

# Punto de operación tau
ax.plot(tau_id, A_step * K_id * 0.632, 'o', color=COLOR_PUNTO, ms=14, zorder=5)
ax.annotate(f'$t = \\tau = {tau_id:.0f}$ s\n$y = {A_step * K_id * 0.632:.3f}$ V',
            xy=(tau_id, A_step * K_id * 0.632),
            xytext=(tau_id + 3, A_step * K_id * 0.632 - 0.15),
            fontsize=11, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('Salida $y$ (V)')
ax.set_title('Identificación de función de transferencia a partir de datos experimentales',
             fontsize=13)
ax.legend(fontsize=10, loc='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 10. Catálogo completo de ejercicios: todos los patrones

| # | Tipo de ejercicio | Dominio | Ecuación clave | Dificultad |
|---|-------------------|---------|---------------|------------|
| 1 | Modelar sistema mecánico (traslación) | Mecánico | $m\ddot{x} + b\dot{x} + kx = F$ | Media |
| 2 | Modelar circuito eléctrico (RLC) | Eléctrico | $LC\ddot{v}_C + RC\dot{v}_C + v_C = v$ | Media |
| 3 | Modelar tanque hidráulico | Hidráulico | $A\dot{h} = Q_i - c\sqrt{h}$ | Media |
| 4 | Modelar sistema térmico | Térmico | $C_{th}\dot{T} = P - K(T - T_{ext})$ | Baja |
| 5 | Linealizar EDO no lineal | General | Taylor + variables incrementales | Alta |
| 6 | Obtener FT desde EDO lineal | General | Laplace + despejar | Media |
| 7 | Simular NL con solve_ivp | Computacional | `solve_ivp(f, t_span, y0)` | Media |
| 8 | Comparar NL vs linealizado | Análisis | Superponer respuestas | Media |
| 9 | Aplicar analogías entre dominios | Conceptual | Tabla de equivalencias | Baja |
| 10 | Identificar FT desde respuesta experimental | Experimental | $K$, $\tau$, $\zeta$, $\omega_n$ | Alta |

### 10.1 Tipo 1: Modelar sistema mecánico (traslación)

**Procedimiento:**
1. Dibujar diagrama de cuerpo libre
2. Aplicar 2ª ley de Newton: $\sum F = m\ddot{x}$
3. Identificar fuerzas: resorte ($-kx$), amortiguador ($-b\dot{x}$), externas ($F$)
4. Aplicar Laplace y obtener $G(s)$

$$\boxed{G(s) = \frac{1}{ms^2 + bs + k}}$$

**Efecto de los parámetros:**
- Si **$m$ aumenta** $\to$ $\omega_n$ disminuye $\to$ respuesta más lenta
- Si **$b$ aumenta** $\to$ $\zeta$ aumenta $\to$ menos oscilaciones (puede pasar de subamortiguado a sobreamortiguado)
- Si **$k$ aumenta** $\to$ $\omega_n$ aumenta, $\zeta$ disminuye $\to$ más rápido pero con más sobreoscilación

#### Ejercicio resuelto: Suspensión de vehículo simplificada

**Datos:** $m = 300$ kg, $k = 12000$ N/m, $b = 1500$ Ns/m, entrada: perfil de carretera $r(t)$.

**Paso 1-2:** $G(s) = \dfrac{1}{300s^2 + 1500s + 12000}$

**Parámetros:**

$$\omega_n = \sqrt{\frac{12000}{300}} = \sqrt{40} = 6.32 \;\text{rad/s}$$

$$\zeta = \frac{1500}{2\sqrt{300 \times 12000}} = \frac{1500}{2 \times 1897} = 0.395$$

**Resultado:** Sistema subamortiguado con sobreoscilación moderada ($\zeta \approx 0.4$). Frecuencia natural $f_n = \omega_n / 2\pi = 1.01$ Hz.

### 10.2 Tipo 2: Modelar circuito eléctrico (RLC)

**Procedimiento:**
1. Aplicar leyes de Kirchhoff (KVL/KCL)
2. Expresar en términos de una única variable
3. Laplace y obtener $G(s)$

$$\boxed{G(s) = \frac{V_C(s)}{V(s)} = \frac{1}{LCs^2 + RCs + 1}}$$

**Efecto de los parámetros:**
- Si **$R$ aumenta** $\to$ $\zeta$ aumenta $\to$ respuesta más amortiguada
- Si **$L$ aumenta** $\to$ $\omega_n$ disminuye, $\zeta$ disminuye $\to$ más lento y más oscilante
- Si **$C$ aumenta** $\to$ $\omega_n$ disminuye, $\zeta$ aumenta $\to$ más lento pero más amortiguado

#### Ejercicio resuelto: Filtro pasa-baja RLC

**Datos:** $R = 50\;\Omega$, $L = 10$ mH, $C = 1\;\mu$F.

$$\omega_n = \frac{1}{\sqrt{LC}} = \frac{1}{\sqrt{10 \times 10^{-3} \times 10^{-6}}} = \frac{1}{\sqrt{10^{-8}}} = 10000 \;\text{rad/s}$$

$$\zeta = \frac{R}{2}\sqrt{\frac{C}{L}} = \frac{50}{2}\sqrt{\frac{10^{-6}}{10^{-2}}} = 25 \times 10^{-2} = 0.25$$

$$\boxed{G(s) = \frac{1}{10^{-8}s^2 + 5 \times 10^{-5}s + 1}}$$

**Resultado:** Sistema subamortiguado. Frecuencia de corte aprox. $f_n = \omega_n / 2\pi = 1591$ Hz.

### 10.3 Tipo 3: Modelar tanque hidráulico y linealizar

**Procedimiento:**
1. Balance de masa: $A\dot{h} = Q_{in} - Q_{out}$
2. Ley de descarga: $Q_{out} = c\sqrt{h}$ (no lineal)
3. Equilibrio: $h_{eq} = (Q_{i,eq}/c)^2$
4. Taylor: $\sqrt{h} \approx \sqrt{h_{eq}} + \frac{\delta h}{2\sqrt{h_{eq}}}$
5. Obtener $G(s)$

$$\boxed{G(s) = \frac{R_h}{\tau s + 1}, \qquad R_h = \frac{2\sqrt{h_{eq}}}{c}, \qquad \tau = A \cdot R_h}$$

**Efecto de los parámetros:**
- Si **$A$ aumenta** $\to$ $\tau$ aumenta $\to$ respuesta más lenta (más inercia)
- Si **$c$ aumenta** $\to$ $R_h$ disminuye, $\tau$ disminuye $\to$ descarga más rápida
- Si **$h_{eq}$ aumenta** $\to$ $R_h$ aumenta $\to$ el sistema se vuelve más lento (la válvula "pesa" menos relativamente)

#### Ejercicio resuelto: Tanque con punto de equilibrio alto

**Datos:** $A = 3$ m$^2$, $c = 0.2$ m$^{5/2}$/s, $Q_{i,eq} = 0.8$ m$^3$/s.

$$h_{eq} = \left(\frac{0.8}{0.2}\right)^2 = 16 \;\text{m}$$

$$R_h = \frac{2\sqrt{16}}{0.2} = \frac{8}{0.2} = 40 \;\text{s/m}^2$$

$$\tau = 3 \times 40 = 120 \;\text{s}$$

$$\boxed{G(s) = \frac{40}{120s + 1}}$$

**Resultado:** Constante de tiempo de 2 minutos. En 5$\tau$ = 10 minutos el sistema alcanza el 99% del nuevo equilibrio.

### 10.4 Tipo 4: Modelar sistema térmico

**Procedimiento:**
1. Balance de energía: $C_{th}\dot{T} = \sum Q_{in} - \sum Q_{out}$
2. Pérdidas por conducción: $Q_{loss} = K(T - T_{ext})$
3. Ya es lineal $\to$ Laplace directamente

$$\boxed{G(s) = \frac{R_{th}}{\tau_{th} s + 1}, \qquad R_{th} = \frac{1}{K}, \qquad \tau_{th} = \frac{C_{th}}{K}}$$

**Efecto de los parámetros:**
- Si **$C_{th}$ aumenta** $\to$ $\tau$ aumenta $\to$ la habitación tarda más en calentarse/enfriarse
- Si **$K$ aumenta** (peor aislamiento) $\to$ $R_{th}$ disminuye, $\tau$ disminuye $\to$ respuesta más rápida pero equilibrio a menor temperatura

#### Ejercicio resuelto: Horno industrial

**Datos:** $C_{th} = 10^4$ J/K, $K = 5$ W/K, $T_{ext} = 20$ °C, $P = 500$ W.

$$R_{th} = \frac{1}{5} = 0.2 \;\text{K/W}$$

$$\tau_{th} = \frac{10^4}{5} = 2000 \;\text{s} = 33.3 \;\text{min}$$

$$T_{eq} = T_{ext} + P \cdot R_{th} = 20 + 500 \times 0.2 = 120 \;\text{°C}$$

$$\boxed{G(s) = \frac{0.2}{2000s + 1}}$$

### 10.5 Tipo 5: Linealizar una EDO no lineal genérica

**Procedimiento general:**
1. Identificar la no linealidad: $f(x)$ (puede ser $\sqrt{x}$, $x^2$, $\sin(x)$, $e^x$, etc.)
2. Calcular la derivada: $f'(x_{eq})$
3. Aproximar: $f(x) \approx f(x_{eq}) + f'(x_{eq}) \cdot \delta x$

**Linealizaciones frecuentes:**

| No linealidad | $f(x_{eq})$ | $f'(x_{eq})$ | Aproximación |
|---------------|-------------|---------------|---------------|
| $\sqrt{x}$ | $\sqrt{x_{eq}}$ | $\dfrac{1}{2\sqrt{x_{eq}}}$ | $\sqrt{x_{eq}} + \dfrac{\delta x}{2\sqrt{x_{eq}}}$ |
| $x^2$ | $x_{eq}^2$ | $2x_{eq}$ | $x_{eq}^2 + 2x_{eq} \cdot \delta x$ |
| $\sin(x)$ | $\sin(x_{eq})$ | $\cos(x_{eq})$ | $\sin(x_{eq}) + \cos(x_{eq}) \cdot \delta x$ |
| $e^x$ | $e^{x_{eq}}$ | $e^{x_{eq}}$ | $e^{x_{eq}}(1 + \delta x)$ |
| $1/x$ | $1/x_{eq}$ | $-1/x_{eq}^2$ | $\dfrac{1}{x_{eq}} - \dfrac{\delta x}{x_{eq}^2}$ |
| $x \cdot y$ | $x_{eq} y_{eq}$ | --- | $x_{eq} y_{eq} + y_{eq} \delta x + x_{eq} \delta y$ |

> **Error frecuente:** olvidar que la linealización de un producto $x \cdot y$ genera **dos términos**, no uno. Hay que aplicar Taylor en ambas variables.

### 10.6 Tipo 6: Obtener FT desde EDO lineal

**Procedimiento:**
1. Tomar la EDO lineal con coeficientes constantes
2. Aplicar Laplace con condiciones iniciales nulas: $\dot{x} \to sX(s)$, $\ddot{x} \to s^2 X(s)$
3. Despejar $G(s) = Y(s)/U(s)$

**Regla general:** si la EDO es $a_n y^{(n)} + \cdots + a_1 \dot{y} + a_0 y = b_m u^{(m)} + \cdots + b_0 u$:

$$\boxed{G(s) = \frac{b_m s^m + \cdots + b_1 s + b_0}{a_n s^n + \cdots + a_1 s + a_0}}$$

#### Ejercicio resuelto

**EDO:** $2\ddot{y} + 3\dot{y} + 5y = 4\dot{u} + u$

$$G(s) = \frac{4s + 1}{2s^2 + 3s + 5}$$

**Polos:** $s = \dfrac{-3 \pm \sqrt{9 - 40}}{4} = \dfrac{-3 \pm j\sqrt{31}}{4} = -0.75 \pm j1.39$

**Cero:** $s = -1/4 = -0.25$

### 10.7 Tipo 7: Simular sistema no lineal con solve_ivp

**Procedimiento en Python:**
1. Definir la función `f(t, y, ...)` que devuelve $\dot{y}$
2. Especificar `t_span`, condición inicial `y0`, y `t_eval`
3. Llamar `solve_ivp(f, t_span, y0, t_eval=t_eval, args=(...))`
4. Graficar `sol.t` vs `sol.y[0]`

**Plantilla:**

```python
def sistema(t, y, param1, param2):
    dydt = f(y[0], param1, param2)  # EDO no lineal
    return [dydt]

sol = solve_ivp(sistema, (0, T_final), [y0],
                t_eval=np.linspace(0, T_final, 500),
                args=(p1, p2), method='RK45')
```

> **Truco:** para sistemas de 2º orden, convertir a sistema de ecuaciones de primer orden: $y_1 = x$, $y_2 = \dot{x}$.

### 10.8 Tipo 8: Comparar respuesta NL vs linealizada

**Procedimiento:**
1. Simular el sistema NL completo con `solve_ivp`
2. Simular el modelo linealizado con `signal.step` (o `signal.lsim`)
3. Superponer ambas respuestas en la misma gráfica
4. Repetir con perturbaciones de distinto tamaño

**Qué observar:**
- **Perturbación pequeña** ($<10\%$): curvas prácticamente superpuestas
- **Perturbación media** ($10-30\%$): discrepancia visible pero tendencia correcta
- **Perturbación grande** ($>50\%$): divergencia significativa, valor final diferente

**Causa del error:** los términos de Taylor de orden $\geq 2$ ya no son despreciables.

### 10.9 Tipo 9: Aplicar analogías entre dominios

**Procedimiento:**
1. Identificar el tipo de sistema original (mecánico, eléctrico, etc.)
2. Consultar la tabla de analogías
3. Sustituir cada variable/parámetro por su equivalente en el otro dominio
4. La FT tiene la **misma estructura**

#### Ejercicio resuelto: Circuito equivalente de un sistema mecánico

**Sistema mecánico:** $2\ddot{x} + 6\dot{x} + 18x = F$

**Equivalente eléctrico (analogía fuerza-tensión):**
- $m = 2$ kg $\to$ $L = 2$ H
- $b = 6$ Ns/m $\to$ $R = 6\;\Omega$
- $k = 18$ N/m $\to$ $1/C = 18 \to C = 1/18$ F

$$\boxed{2\frac{d^2q}{dt^2} + 6\frac{dq}{dt} + 18q = v(t)}$$

La función de transferencia es idéntica: $G(s) = \dfrac{1}{2s^2 + 6s + 18}$

### 10.10 Tipo 10: Identificación de caja negra desde respuesta experimental

**Procedimiento para sistema de 1er orden (sin sobreoscilación):**
1. Medir la ganancia estática: $K = y_{\infty} / A$ (amplitud del escalón)
2. Medir $\tau$ como el tiempo en que la salida alcanza el 63.2% de $y_{\infty}$
3. $G(s) = K / (\tau s + 1)$

**Procedimiento para sistema de 2º orden (con sobreoscilación):**
1. Medir $K = y_{\infty} / A$
2. Medir la sobreoscilación: $M_p = (y_{max} - y_{\infty}) / y_{\infty}$
3. Calcular $\zeta$: $M_p = e^{-\pi\zeta / \sqrt{1-\zeta^2}}$
4. Medir el periodo de oscilación $T_d$ y calcular $\omega_d = 2\pi / T_d$
5. Calcular $\omega_n = \omega_d / \sqrt{1-\zeta^2}$
6. $G(s) = K\omega_n^2 / (s^2 + 2\zeta\omega_n s + \omega_n^2)$

> **Error frecuente:** confundir $M_p$ (en tanto por uno) con el porcentaje. Si $y_{max} = 1.2$ y $y_{\infty} = 1.0$, entonces $M_p = 0.2$ (20%), NO $M_p = 1.2$.

---

## 11. Resumen y tablas de fórmulas clave

### 11.1 Modelos por dominio físico

| Dominio | Sistema | EDO | FT |
|---------|---------|-----|----|
| Mecánico | Masa-resorte-amort. | $m\ddot{x} + b\dot{x} + kx = F$ | $\dfrac{1}{ms^2 + bs + k}$ |
| Eléctrico | RLC serie ($v_C$) | $LC\ddot{v}_C + RC\dot{v}_C + v_C = v$ | $\dfrac{1}{LCs^2 + RCs + 1}$ |
| Hidráulico | Tanque + válvula | $A\dot{h} = Q_i - c\sqrt{h}$ | $\dfrac{R_h}{\tau s + 1}$ (linealizado) |
| Térmico | Habitación + calefactor | $C_{th}\dot{T} = P - K(T - T_{ext})$ | $\dfrac{R_{th}}{\tau_{th} s + 1}$ |

### 11.2 Linealizaciones más usadas

| Función | Linealización alrededor de $x_{eq}$ |
|---------|--------------------------------------|
| $\sqrt{x}$ | $\sqrt{x_{eq}} + \dfrac{\delta x}{2\sqrt{x_{eq}}}$ |
| $\sin(x)$ | $\sin(x_{eq}) + \cos(x_{eq}) \cdot \delta x$ |
| $x^2$ | $x_{eq}^2 + 2x_{eq} \cdot \delta x$ |
| $e^x$ | $e^{x_{eq}}(1 + \delta x)$ |

### 11.3 Procedimiento de 7 pasos (resumen)

$$\boxed{\text{EDO NL} \xrightarrow{\text{equilibrio}} \text{Taylor} \xrightarrow{\text{1er orden}} \text{EDO lineal} \xrightarrow{\text{Laplace}} G(s)}$$

### 11.4 Parámetros clave de identificación

| Parámetro | 1er orden | 2º orden |
|-----------|-----------|----------|
| Ganancia $K$ | $y_{\infty} / A$ | $y_{\infty} / A$ |
| Constante de tiempo | $\tau$ = tiempo al 63.2% | --- |
| Amortiguamiento $\zeta$ | --- | De $M_p = e^{-\pi\zeta/\sqrt{1-\zeta^2}}$ |
| Frec. natural $\omega_n$ | --- | De $\omega_d = 2\pi / T_d$ y $\omega_n = \omega_d / \sqrt{1-\zeta^2}$ |